In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_joined, basic_clean

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

# Home Credit Default Risk: Targeted EDA

This notebook explores the Home Credit dataset through the lens of three stakeholder questions:
1. **Chief Credit Officer:** what does this portfolio look like and how does default rate vary?
2. **Head of Compliance:** how does default rate vary across protected-class proxies?
3. **Head of Growth:** what's the approval/decision picture and what segments drive volume?

The goal is not exhaustive coverage of all 130+ features. The goal is to surface the
small number of facts that will drive modeling decisions in the next milestones.

## Section 1: What does this portfolio look like?

Written for the Chief Credit Officer: the basic shape of the book.

In [ ]:
df = load_joined()
df = basic_clean(df)
print(f"Shape: {df.shape}")
print(f"Default rate: {df['TARGET'].mean():.4f}")

In [ ]:
# Top-line stats
n_apps = len(df)
default_rate = df["TARGET"].mean()
n_defaults = df["TARGET"].sum()

print(f"Total applications: {n_apps:,}")
print(f"Defaults (TARGET=1): {n_defaults:,} ({default_rate:.2%})")
print(f"Non-defaults: {n_apps - n_defaults:,}")

In [ ]:
# Loan type breakdown
contract_summary = df.groupby("NAME_CONTRACT_TYPE").agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
contract_summary["pct_of_book"] = contract_summary["count"] / n_apps
print(contract_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loan amount (linear is fine, less skewed)
axes[0].hist(df["AMT_CREDIT"], bins=50, color="steelblue")
axes[0].set_xlabel("Loan amount")
axes[0].set_title("Loan amount distribution")
axes[0].set_xlim(0, df["AMT_CREDIT"].quantile(0.99))

# Income on log scale — clip to non-zero and reasonable range
income = df["AMT_INCOME_TOTAL"]
income = income[(income > 0) & (income <= income.quantile(0.99))]  # drop top 1%
axes[1].hist(income, bins=50, color="steelblue")
axes[1].set_xscale("log")
axes[1].set_xlabel("Annual income (log scale, 99th pct truncated)")
axes[1].set_title("Income distribution")

plt.tight_layout()
plt.show()

print(f"Median loan amount: {df['AMT_CREDIT'].median():,.0f}")
print(f"Median income: {df['AMT_INCOME_TOTAL'].median():,.0f}")
print(f"Income range: {income.min():,.0f} to {income.max():,.0f}")
print(f"Median loan-to-income: {(df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']).median():.2f}x")

In [ ]:
# Default rate by income decile
df["income_decile"] = pd.qcut(df["AMT_INCOME_TOTAL"], 10, labels=False, duplicates="drop")
income_default = df.groupby("income_decile")["TARGET"].mean()
print("Default rate by income decile (low → high):")
print(income_default.round(4))

# Also: ratio of top to bottom decile
print(f"\nTop decile default rate: {income_default.iloc[-1]:.4f}")
print(f"Bottom decile default rate: {income_default.iloc[0]:.4f}")
print(f"Ratio: {income_default.iloc[0] / income_default.iloc[-1]:.2f}x")

In [ ]:
# Payment-to-income ratio
df["pti"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["pti_decile"] = pd.qcut(df["pti"], 10, labels=False, duplicates="drop")
pti_default = df.groupby("pti_decile")["TARGET"].mean()
print("Default rate by payment-to-income decile (low PTI → high PTI):")
print(pti_default.round(4))

**Section 1 takeaways for the CCO:**

- Portfolio of 307,511 applications with an 8.07% default rate. Heavily cash-loan
  oriented (90.5%); revolving loans (9.5%) default at notably lower rates (~35%
  less), likely a selection effect from credit-line origination.

- Median loan size 513,531 against median income 147,150 (3.27x loan-to-income).
  Loan amounts cluster around standard product sizes (~250k, ~500k currency units),
  suggesting Home Credit offers fixed product tiers rather than fully customized
  originations.

- Raw income is a surprisingly weak signal. Default rate by income decile is
  non-monotonic — middle deciles actually default at higher rates (~9%) than the
  bottom (~8.2%), and the top decile drops to 6.1%. Total range is only 1.33x.
  Payment-to-income (PTI) is more monotonic and somewhat stronger, but still
  modest (1.24x across most of the distribution). Real signal here will come
  from interactions, not raw income.

- **Data quality flag:** AMT_INCOME_TOTAL has extreme outliers (max 117M units,
  vs. median 147k). These are almost certainly entry errors. Will need to cap or
  log-transform in modeling.

## Section 2: Who is the thin-file segment?

The `is_thin_file` flag is defined as `bureau_count <= 1`, capturing applicants with
zero or one prior bureau record (~26% of the book). This section examines the
relationship between bureau history and default risk, justifies the cutoff choice,
and considers what the broader bureau-count distribution tells us about risk.

In [ ]:
# Distribution of bureau record counts
bureau_dist = df["bureau_count"].value_counts().sort_index()
print("Bureau count distribution (top 15 values):")
print(bureau_dist.head(15))
print(f"\nApplicants with 0 bureau records: {(df['bureau_count'] == 0).sum():,} ({(df['bureau_count'] == 0).mean():.2%})")

In [ ]:
# Default rate by bureau count bin
df["bureau_count_bin"] = pd.cut(
    df["bureau_count"],
    bins=[-0.1, 0, 1, 2, 4, 8, 16, np.inf],
    labels=["0", "1", "2", "3-4", "5-8", "9-16", "17+"],
)

bureau_default = df.groupby("bureau_count_bin", observed=True).agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
bureau_default["pct_of_book"] = bureau_default["count"] / len(df)
print(bureau_default)

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
bureau_default["default_rate"].plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(default_rate, color="red", linestyle="--", label=f"Overall {default_rate:.2%}")
ax.set_ylabel("Default rate")
ax.set_xlabel("Bureau record count")
ax.set_title("Default rate by bureau record count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Try alternative thin-file cutoffs and see what fraction each captures
for cutoff in [0, 1, 2, 3, 4, 5]:
    pct = (df["bureau_count"] <= cutoff).mean()
    default_in_segment = df.loc[df["bureau_count"] <= cutoff, "TARGET"].mean()
    default_out_segment = df.loc[df["bureau_count"] > cutoff, "TARGET"].mean()
    lift = default_in_segment / default_out_segment
    print(
        f"Cutoff <= {cutoff}: captures {pct:.1%} of book | "
        f"default rate in: {default_in_segment:.2%} | "
        f"out: {default_out_segment:.2%} | "
        f"lift: {lift:.2f}x"
    )

**Section 2 takeaways:**

- Default rate by bureau record count is non-monotonic — U-shaped. Both the
  zero-bureau group (10.12%) and the 17+ group (9.54%) have elevated default
  relative to the 3-8 range (7.3-7.4%).

- The U-shape is itself a more interesting finding than any single cutoff:
  borrowers with NO bureau history are risky (no track record); borrowers with
  EXTENSIVE bureau history (17+) are also risky (likely credit-hungry or
  debt-cycling). The middle is safest. A real credit model should treat this
  as non-linear, not as "more bureau = better."

**Decision: defining thin-file as `bureau_count <= 1`** (26.0% of the book,
1.24x default lift vs. the rest). Reasoning:
- Stronger and more meaningful lift than wider cutoffs (`<= 2` gives only 1.18x).
- Cleaner business narrative: "zero or one prior bureau record" = applicants with
  essentially no track record visible to the bureau. Higher cutoffs lack a
  principled threshold.
- Segment size (~80k applicants) is comfortable for separate segment modeling in
  Milestone 3.
- The tightest cutoff (`== 0`) has the cleanest narrative and strongest lift but
  the segment may be too small for stable separate modeling. We can revisit if
  the segment model on `<= 1` performs unexpectedly.

## Section 3: What signal exists?

Targeted check of the features expected to carry the most signal. The point is not to
plot everything — it's to confirm what should matter does, surface anything unexpected,
and flag the EXT_SOURCE features for a conceptual discussion later.

In [ ]:
# EXT_SOURCE: the three external scores
# These will likely dominate any model trained on this data.

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

print("Coverage (% non-missing):")
for col in ext_cols:
    print(f"  {col}: {df[col].notna().mean():.1%}")

print("\nMean value by TARGET:")
print(df.groupby("TARGET")[ext_cols].mean().round(3))

In [ ]:
# Default rate by EXT_SOURCE decile
# If these features carry signal, default rate should drop monotonically as score increases.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(ext_cols):
    # Only use non-missing rows for the decile analysis
    sub = df[df[col].notna()].copy()
    sub["decile"] = pd.qcut(sub[col], 10, labels=False, duplicates="drop")
    decile_default = sub.groupby("decile")["TARGET"].mean()

    axes[i].bar(decile_default.index, decile_default.values, color="steelblue")
    axes[i].axhline(default_rate, color="red", linestyle="--", alpha=0.6)
    axes[i].set_title(f"{col} (n={len(sub):,})")
    axes[i].set_xlabel("Score decile (low → high)")
    axes[i].set_ylabel("Default rate")

plt.suptitle("Default rate by EXT_SOURCE decile", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Age (DAYS_BIRTH): convert to years and bin
df["age_years"] = -df["DAYS_BIRTH"] / 365

age_bins = pd.cut(df["age_years"], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 70])
age_default = df.groupby(age_bins, observed=True).agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
age_default["pct_of_book"] = age_default["count"] / len(df)
print(age_default)

fig, ax = plt.subplots(figsize=(10, 4))
age_default["default_rate"].plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(default_rate, color="red", linestyle="--", label=f"Overall {default_rate:.2%}")
ax.set_ylabel("Default rate")
ax.set_xlabel("Age (years)")
ax.set_title("Default rate by age band")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Employment tenure (DAYS_EMPLOYED)
# Recall: 365243 is a sentinel for unemployed/retired — basic_clean already replaced with NaN.
df["employment_years"] = -df["DAYS_EMPLOYED"] / 365

# Look at the distribution among employed applicants only
employed = df[df["employment_years"].notna()].copy()
print(f"Applicants with employment data: {len(employed):,} ({len(employed)/len(df):.1%})")
print(f"Applicants without (unemployed/retired): {len(df) - len(employed):,}")

# Default rate by tenure bin
tenure_bins = pd.cut(employed["employment_years"], bins=[0, 1, 2, 5, 10, 20, 50])
tenure_default = employed.groupby(tenure_bins, observed=True).agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
print(tenure_default)

# Also: what's the default rate among the unemployed/retired group?
unemployed_default = df.loc[df["employment_years"].isna(), "TARGET"].mean()
print(f"\nDefault rate among unemployed/retired: {unemployed_default:.2%}")
print(f"Default rate among employed: {employed['TARGET'].mean():.2%}")

In [ ]:
# Education
edu_default = df.groupby("NAME_EDUCATION_TYPE").agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4).sort_values("default_rate", ascending=False)
edu_default["pct_of_book"] = edu_default["count"] / len(df)
print(edu_default)

In [ ]:
# Occupation (only show categories with at least 1,000 applicants for stability)
occ_default = df.groupby("OCCUPATION_TYPE").agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
occ_default = occ_default[occ_default["count"] >= 1000].sort_values("default_rate", ascending=False)
occ_default["pct_of_book"] = occ_default["count"] / len(df)
print(occ_default)

In [ ]:
# Gender: raw default rate by gender
# This is the descriptive number. The actual fair lending analysis happens after modeling.
gender_default = df.groupby("CODE_GENDER").agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
gender_default["pct_of_book"] = gender_default["count"] / len(df)
print(gender_default)

**Section 3 takeaways:**

- All three EXT_SOURCE scores show strong monotonic relationships with default rate.
  EXT_SOURCE_3 is the steepest (~6-7x ratio between top and bottom deciles); 
  EXT_SOURCE_2 has near-universal coverage (99.8%) and will be the model workhorse;
  EXT_SOURCE_1 has weakest signal AND lowest coverage — addressed in the missingness analysis below.
- Age shows the textbook pattern: ~12% default at 20-25, declining to ~5% at 60-70.
  Strong signal but a fair-lending sensitivity to flag.
- Education is a 6x gradient from Lower secondary (11%) to Academic degree (2%);
  Occupation is even sharper at the extremes (17% for Low-skill Laborers, 5% for
  Accountants). Both likely confounded with income and socioeconomic status.
- **Surprising finding:** unemployed/retired applicants default at 5.4%, *lower* than
  the 8.7% for employed applicants. The "missing employment data" group is mostly
  retirees, who are actually low-risk. Treating DAYS_EMPLOYED missingness as a
  category, not noise, is important.
- Raw gender default rate gap: 7.0% (F) vs 10.1% (M), a 1.45x ratio. Descriptive
  only; the fair-lending analysis in Milestone 4 will look at model-driven outcomes.

## Section 4: Where is data missing, and is it informative?

For each feature with non-trivial missingness, check whether default rate differs
between applicants with and without that value. If the gap is meaningful, missingness
itself carries signal and must be preserved in modeling (typically via a missingness
indicator flag, or by using a model like LightGBM that handles NaN natively).

In [ ]:
# Compute missingness rate for every column
missing_rates = df.isna().mean().sort_values(ascending=False)

# Show columns with at least 1% missing, top 30
print("Top features by missingness rate:")
print(missing_rates[missing_rates > 0.01].head(30).round(3))

In [ ]:
# For features with meaningful missingness (>5%), compute default rate
# among missing vs non-missing rows.
# Focus on the features we already know matter, not exhaustive.

features_to_check = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_EMPLOYED",
    "OCCUPATION_TYPE",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
    "OWN_CAR_AGE",
    "bureau_credit_sum_mean",  # NaN for applicants with no bureau records
]

results = []
for feat in features_to_check:
    if feat not in df.columns:
        continue
    miss_mask = df[feat].isna()
    n_missing = miss_mask.sum()
    if n_missing == 0:
        continue
    rate_missing = df.loc[miss_mask, "TARGET"].mean()
    rate_present = df.loc[~miss_mask, "TARGET"].mean()
    lift = rate_missing / rate_present if rate_present > 0 else float("nan")
    results.append({
        "feature": feat,
        "pct_missing": miss_mask.mean(),
        "default_rate_missing": rate_missing,
        "default_rate_present": rate_present,
        "lift": lift,
    })

missingness_df = pd.DataFrame(results).round(4).sort_values("lift", ascending=False)
print(missingness_df.to_string(index=False))

In [ ]:
# Visualize the key result
fig, ax = plt.subplots(figsize=(10, 5))
missingness_df_sorted = missingness_df.sort_values("lift")

colors = ["steelblue" if l < 1 else "indianred" for l in missingness_df_sorted["lift"]]
ax.barh(missingness_df_sorted["feature"], missingness_df_sorted["lift"], color=colors)
ax.axvline(1.0, color="black", linestyle="--", alpha=0.5, label="lift = 1.0")
ax.set_xlabel("Default rate ratio (missing / present)")
ax.set_title("Is missingness informative?")
ax.legend()
plt.tight_layout()
plt.show()

**Section 4 takeaways:**

- Missingness in this dataset is highly informative, not noise. Two distinct patterns:
  - **Higher default when missing** (lift 1.14-1.34x): AMT_REQ_CREDIT_BUREAU_YEAR,
    bureau_credit_sum_mean, EXT_SOURCE_3, OWN_CAR_AGE, EXT_SOURCE_1. These all share
    a common cause: the applicant is invisible to bureau/scoring infrastructure, which
    is itself a credit risk signal.
  - **Lower default when missing** (lift 0.62-0.74x): DAYS_EMPLOYED, OCCUPATION_TYPE.
    Reflects the retiree population, who have stable pension income.

- The "missing" sentinel value 365243 in DAYS_EMPLOYED (already handled in basic_clean)
  is the most striking single example: treating it as a number would massively distort
  the model; treating it as missing recovers the retiree segment as a low-risk group.

- Modeling implication: do not naively impute missing values. LightGBM handles NaN
  natively; the logistic regression baseline will use WoE encoding which treats missing
  as its own bin. Both preserve the missingness signal.

- Section ignored the 47 housing/apartment columns with 50%+ missingness — these are
  regional aggregate statistics with weak signal; we won't use them in modeling.

## Section 5: Fair lending pre-check

Establish baseline default rates across the two protected-class proxies available in
this dataset: gender (CODE_GENDER) and age (DAYS_BIRTH). This is descriptive, not
diagnostic — the actual disparate impact analysis happens in Milestone 4, where we
look at model-driven approval rates and error rates by group.

The relevant US regulatory frame: ECOA / Reg B prohibits credit decisions based on
gender, and age over 62 is partially protected. Even where these features are not
directly used, models can produce disparate outcomes through correlated features
("disparate impact"). The four-fifths rule says approval rate for a protected group
must be at least 80% of the rate for the reference group.

Note: Home Credit does not include race data. Real US fair lending analysis would
include race; here we work with what's available.

In [ ]:
# Gender baseline (already seen in Section 3, formalized here)
gender_default = df.groupby("CODE_GENDER").agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
gender_default["pct_of_book"] = gender_default["count"] / len(df)

# Compute lift relative to the lower-default group (F)
ref_rate = gender_default.loc["F", "default_rate"]
gender_default["lift_vs_F"] = (gender_default["default_rate"] / ref_rate).round(2)

print("Default rate by gender:")
print(gender_default)

In [ ]:
# Age baseline — using broader bins for fair lending framing
# Highlight the 62+ band specifically since age 62+ is partially protected under ECOA
df["age_band_fair_lending"] = pd.cut(
    df["age_years"],
    bins=[20, 25, 35, 45, 55, 62, 70],
    labels=["20-25", "25-35", "35-45", "45-55", "55-62", "62+"],
)

age_default_fl = df.groupby("age_band_fair_lending", observed=True).agg(
    count=("TARGET", "size"),
    default_rate=("TARGET", "mean"),
).round(4)
age_default_fl["pct_of_book"] = age_default_fl["count"] / len(df)

# Reference rate = the band with the lowest default rate
ref_rate = age_default_fl["default_rate"].min()
age_default_fl["lift_vs_best"] = (age_default_fl["default_rate"] / ref_rate).round(2)

print("Default rate by age band:")
print(age_default_fl)

In [ ]:
# Do the EXT_SOURCE features themselves vary systematically by gender?
# This is relevant because EXT_SOURCE will dominate the model — if these external
# scores are systematically lower for one group, that bakes any historical disparity
# into the new model.

print("Mean EXT_SOURCE by gender:")
ext_by_gender = df.groupby("CODE_GENDER")[ext_cols].mean().round(3)
print(ext_by_gender)

# Same for age
print("\nMean EXT_SOURCE by age band:")
ext_by_age = df.groupby("age_band_fair_lending", observed=True)[ext_cols].mean().round(3)
print(ext_by_age)

In [ ]:
# Quick visualization of the gender gap in EXT_SOURCE_2 (the most universal feature)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EXT_SOURCE_2 distribution by gender
for gender in ["F", "M"]:
    axes[0].hist(
        df.loc[df["CODE_GENDER"] == gender, "EXT_SOURCE_2"].dropna(),
        bins=50,
        alpha=0.5,
        label=gender,
        density=True,
    )
axes[0].set_xlabel("EXT_SOURCE_2")
axes[0].set_ylabel("Density")
axes[0].set_title("EXT_SOURCE_2 distribution by gender")
axes[0].legend()

# EXT_SOURCE_2 by age band
ext2_by_age = df.groupby("age_band_fair_lending", observed=True)["EXT_SOURCE_2"].mean()
ext2_by_age.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_ylabel("Mean EXT_SOURCE_2")
axes[1].set_xlabel("Age band")
axes[1].set_title("Mean EXT_SOURCE_2 by age band")

plt.tight_layout()
plt.show()

**Section 5 takeaways:**

- Raw gender default gap: F 7.00% vs M 10.14% (1.45x lift). At zero-threshold (approve
  everyone), four-fifths ratio is 0.97 — passes easily. The question is what happens
  at a real decision threshold.

- Raw age default gradient: 20-25 (12.30%) defaults at 2.58x the rate of 62+ (4.76%).
  This is the steepest fair-lending-sensitive gradient in the data. Age 62+ is
  partially protected under ECOA. Four-fifths ratio between extremes at zero-threshold
  is 0.92 — passes, but a real threshold will compress this significantly.

- **Critical finding: EXT_SOURCE_1 is heavily demographic.** Women score 0.546 vs men
  0.407 (0.14 gap). 20-25 year olds score 0.282 vs 62+ at 0.750 (2.7x gradient). 
  EXT_SOURCE_2 and 3 show no gender disparity and much milder age gradients. This 
  raises a real modeling decision: using EXT_SOURCE_1 imports its demographic structure
  into the new model. Worth flagging in the blog post — most projects don't engage
  with this kind of upstream-model question.

- For fair lending: the descriptive baseline says the raw data passes the four-fifths
  rule trivially. The interesting question is whether the *model* at the *chosen 
  threshold* maintains that. Milestone 4 will measure this.

## Section 6: What I'm taking forward

A summary of the EDA findings that will drive modeling decisions in Milestones 2-4.

**What this portfolio looks like.** 307,511 loan applications with an 8.07% overall
default rate, 90% cash loans and 10% revolving, with revolving customers defaulting
at ~35% lower rates. Median loan-to-income ratio of 3.27x. Loan amounts cluster
around standard product sizes (~250k, ~500k currency units), suggesting fixed
product tiers rather than fully customized originations.

**Thin-file segment definition.** Defined as `bureau_count <= 1` (zero or one prior
bureau record), capturing 26% of the book with a 1.24x default lift over the rest.
The bureau-count-to-default relationship is non-monotonic: both the zero-bureau
group (10.12% default) and the 17+ bureau group (9.54%) have elevated risk
relative to the 3-8 range (~7.3%). Real credit models should treat bureau
history as a non-linear signal.

**Likely model drivers.** EXT_SOURCE_1/2/3 (external credit scores) carry the
strongest signal, with default rates dropping monotonically across deciles by
6-7x. Age, employment tenure, education, and occupation are secondary but
substantial predictors. EXT_SOURCE_2 has near-universal coverage (99.8%) and
will be the model workhorse.

**Missingness is signal, not noise.** Eight features show meaningful missingness
patterns: applicants invisible to bureau or scoring infrastructure default at
elevated rates (lift 1.14-1.34x); applicants without employment data are mostly
retirees and default at *lower* rates (lift 0.62x). Both LightGBM (native NaN
handling) and WoE-encoded logistic regression preserve this signal; naive mean
imputation would destroy it.

**Fair lending considerations.** Raw default rates differ meaningfully by
protected-class proxies: gender (F 7.0% vs M 10.1%, 1.45x lift) and age (20-25
band defaults at 2.58x the 62+ band). EXT_SOURCE_1 specifically embeds heavy
demographic structure — women score 0.546 vs men 0.407 on average, and the score
varies by 2.7x across age bands. EXT_SOURCE_2 and 3 don't show similar
disparity. Using EXT_SOURCE_1 in the model would import its demographic patterns;
this is a real modeling decision to engage with in Milestone 4.

**Features deliberately excluded from modeling.**
- 47 housing/apartment columns (e.g., `COMMONAREA_AVG`, `LIVINGAPARTMENTS_MODE`):
  regional aggregates with 50-70% missingness and weak signal.
- 20 `FLAG_DOCUMENT_*` columns: most show no individual signal.
- Social circle features (`OBS_30_CNT_SOCIAL_CIRCLE`, `DEF_30_CNT_SOCIAL_CIRCLE`):
  ethically questionable and weak signal.

**Open questions for downstream work.**
1. Use or exclude EXT_SOURCE_1 given its demographic structure?
2. Does a separate segment model for the thin-file population (`<= 1` bureau record,
  26% of book) outperform a single pooled model?
3. At the chosen decision threshold, does the model satisfy the four-fifths rule
  on gender and age?